# Poincare section with particle selection

Visualization of the completed **48-particle, 5,000-cycle, 20-step/cycle BM4Implicit** study. This notebook only reads saved results. It does not integrate trajectories or access AWS.

The figure includes all 240,000 saved returns, excluding cycle zero. Coordinates are wrapped into the periodic cell and normalized by its length L. Original fixed particle colors are preserved. The saved study does not include a reference trajectory.

Use checkboxes to show any subset, **Only** to isolate one particle, and **Show all**, **Hide all**, or **Invert** to change the selection. Export the visible plot as PNG. The generated HTML is self-contained and works offline.

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import IFrame, FileLink, display
from visualization.poincare_selector import export_poincare_selector

SOURCE_STUDY = Path('../Poincare_BM4_48_radiales_5000_ciclos_20_steps_16_procesos_spot')
RUN_ID = 'aws_48p_5000c_20s_16proc_spot_20260920'
SOURCE_RESULTS = SOURCE_STUDY / 'resultados' / RUN_ID
OUTPUT_HTML = Path('poincare_selector.html')
EXPECTED_PARTICLES = 48
EXPECTED_CYCLES = 5000
EXPECTED_STEPS_PER_CYCLE = 20


In [2]:
manifest = json.loads((SOURCE_RESULTS / 'COMPLETE.json').read_text())
assert manifest['run_id'] == RUN_ID
for name in ('metadata.json', 'positions_after_each_cycle.csv'):
    with (SOURCE_RESULTS / name).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == manifest['sha256'][name]
metadata = json.loads((SOURCE_RESULTS / 'metadata.json').read_text())
assert (metadata['particle_count'], metadata['cycles'], metadata['steps_per_cycle']) == (
    EXPECTED_PARTICLES, EXPECTED_CYCLES, EXPECTED_STEPS_PER_CYCLE)
positions = pd.read_csv(SOURCE_RESULTS / 'positions_after_each_cycle.csv', float_precision='round_trip')
positions = positions.sort_values(['cycle', 'particle'])
particle_ids = np.arange(1, EXPECTED_PARTICLES + 1)
np.testing.assert_array_equal(positions['cycle'], np.repeat(np.arange(1, EXPECTED_CYCLES + 1), EXPECTED_PARTICLES))
np.testing.assert_array_equal(positions['particle'], np.tile(particle_ids, EXPECTED_CYCLES))
colors = [metadata['colours'][str(p)] for p in particle_ids]
np.testing.assert_array_equal(positions['color'], np.tile(colors, EXPECTED_CYCLES))
# Shape: (cycle excluding zero, particle, normalized x/y coordinate).
xy_over_L = positions[['x_over_L', 'y_over_L']].to_numpy().reshape(EXPECTED_CYCLES, EXPECTED_PARTICLES, 2)
print(f'Loaded {len(positions):,} verified return points; no integration performed.')


Loaded 240,000 verified return points; no integration performed.


In [3]:
export_poincare_selector(
    OUTPUT_HTML, xy_over_L, particle_ids, colors,
    title='Hamiltonian Poincare section | BM4Implicit',
    subtitle=f"{EXPECTED_PARTICLES} particles · {EXPECTED_CYCLES} cycles · {EXPECTED_STEPS_PER_CYCLE} steps/cycle · {metadata['process_count']} processes",
)
display(FileLink(str(OUTPUT_HTML)))
display(IFrame(str(OUTPUT_HTML), width='100%', height=1150))
print(f'Interactive Poincare selector URL: {OUTPUT_HTML.resolve().as_uri()}')


/home/juan/Proyectos/GC2D_intranet/notebooks/developements/poincare_section/Poincare_BM4_48_selector_particulas/poincare_selector.html